# Data Pipeline: Build Universe and Historical Price Tables

This notebook reconstructs the upstream data pipeline for the project.

It performs three tasks:

1. Build the base company universe from SEC `company_tickers.json`
2. Prepare the enriched company universe used in downstream analysis
3. Download historical market price data from Yahoo Finance and export structured tables

These exported tables are then used in the clustering and Y-label construction modules.

## 1. Import libraries

This cell imports the Python libraries used for JSON parsing, data cleaning, and market data downloading.

In [2]:
import json
import math
import time
from datetime import datetime

import pandas as pd
import yfinance as yf

## 2. Load SEC company mapping

This cell reads the SEC official company mapping file (`company_tickers.json`) and converts it into a tabular company universe with CIK, ticker, and company name.

In [3]:
with open("company_tickers.json", "r", encoding="utf-8") as f:
    data = json.load(f)

base_universe = pd.DataFrame.from_dict(data, orient="index")

base_universe = base_universe.rename(columns={
    "cik_str": "CIK",
    "ticker": "ticker",
    "title": "company_name"
})

base_universe["ticker"] = base_universe["ticker"].astype(str).str.upper().str.strip()
base_universe["CIK"] = base_universe["CIK"].astype(str).str.zfill(10)

base_universe = base_universe[["ticker", "company_name", "CIK"]].drop_duplicates().reset_index(drop=True)

print("Base universe shape:", base_universe.shape)
display(base_universe.head())

Base universe shape: (10215, 3)


,ticker,company_name,CIK
0,NVDA,NVIDIA CORP,0001045810
1,AAPL,Apple Inc.,0000320193
2,MSFT,MICROSOFT CORP,0000789019
3,GOOGL,Alphabet Inc.,0001652044
4,AMZN,AMAZON COM INC,0001018724


## 3. Export the SEC base universe

This cell saves the SEC-based company universe as an intermediate file for reproducibility.

In [4]:
base_universe.to_csv("base_universe_from_sec.csv", index=False)
print("Saved: base_universe_from_sec.csv")

Saved: base_universe_from_sec.csv


## 4. Load the enriched company universe

The company universe used in this project is constructed based on the SEC official company mapping file `company_tickers.json`, which provides the CIK–ticker–company name relationship.

Building on this standardized base, we use a preprocessed dataset `primary_ticker_only.csv`, which has been cleaned and enriched in prior steps. Specifically, this dataset is constructed by:

- removing firms with missing or invalid ticker and SIC information  
- enriching each firm with SIC classification and SIC descriptions  
- filtering the dataset to focus on technology-related firms based on SIC categories  
- selecting one primary ticker per company (CIK) to ensure a one-to-one mapping between firm and ticker  

This preprocessing step standardizes the company universe and ensures consistency across data sources. Compared with the raw SEC mapping, this dataset already includes structured firm-level metadata required for downstream analysis.

This file serves as the starting point for subsequent data processing, including price data retrieval and event-based analysis.

In [ ]:
df = pd.read_csv("primary_ticker_only.csv")

print("primary_ticker_only.csv shape:", df.shape)
print(df.columns.tolist())
display(df.head())

## 5. Standardize the company universe schema

This cell renames columns into a consistent schema and creates a structured company table for downstream merging.

In [ ]:
df_clean = df.rename(columns={
    "ticker": "ticker",
    "title": "company_name",
    "CIK": "cik",
    "SIC": "sic",
    "SIC_description": "sic_description"
}).copy()

df_clean["ticker"] = df_clean["ticker"].astype(str).str.upper().str.strip()
df_clean["cik"] = df_clean["cik"].astype(str).str.zfill(10)

df_clean = df_clean.reset_index(drop=True)
df_clean.insert(0, "company_id", df_clean.index + 1)

company_out = df_clean[["company_id", "cik", "ticker", "company_name", "sic", "sic_description"]].copy()

print("Structured company table shape:", company_out.shape)
display(company_out.head())

## 6. Export the structured company table

This cell exports the cleaned company table as `company.csv`, which will be used in later event-study and modeling notebooks.

In [ ]:
company_out.to_csv("company.csv", header=False, index=False)
print("Saved: company.csv")

## 7. Define the Yahoo Finance download window and a reusable price download function

This cell sets the fixed historical window for downloading stock price data.

In [ ]:
FIXED_START = "2015-11-28"
TODAY = datetime.today().strftime("%Y-%m-%d")

print("Download start:", FIXED_START)
print("Download end:", TODAY)

This cell defines a helper function that downloads historical OHLCV data for one ticker from Yahoo Finance and standardizes the output columns.

In [ ]:
def fetch_price_fixed_window(ticker: str):
    try:
        dfp = yf.download(
            ticker,
            start=FIXED_START,
            end=TODAY,
            auto_adjust=False,
            progress=False
        )

        if dfp is None or dfp.empty:
            return None

        if isinstance(dfp.columns, pd.MultiIndex):
            try:
                dfp = dfp.xs(ticker, axis=1, level=1)
            except Exception:
                dfp.columns = [c[0] for c in dfp.columns]

        dfp = dfp.reset_index().rename(columns={
            "Date": "trade_date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume"
        })

        dfp = dfp[["trade_date", "open", "high", "low", "close", "adj_close", "volume"]]
        return dfp

    except Exception:
        return None

## 8. Download historical price data for all firms

This cell loops through the company universe, downloads historical prices ticker by ticker, and stores them in a row-level list before converting to a structured table.

In [ ]:
df_comp = company_out[["company_id", "ticker"]].drop_duplicates()
daily_rows = []

for _, row in df_comp.iterrows():
    cid = int(row["company_id"])
    ticker = str(row["ticker"]).strip().upper()

    print(f"Fetching {ticker} (company_id={cid}) ...")
    dfp = fetch_price_fixed_window(ticker)

    if dfp is None:
        print(f"Warning: {ticker} has no price data.")
        continue

    for rec in dfp.itertuples(index=False, name=None):
        trade_date_raw, open_raw, high_raw, low_raw, close_raw, adj_close_raw, vol_raw = rec

        trade_date = pd.to_datetime(trade_date_raw).strftime("%Y-%m-%d")

        def to_num(x):
            if x is None:
                return None
            if isinstance(x, float) and math.isnan(x):
                return None
            return float(x)

        open_ = to_num(open_raw)
        high_ = to_num(high_raw)
        low_ = to_num(low_raw)
        close_ = to_num(close_raw)
        adj_close_ = to_num(adj_close_raw)

        if vol_raw is None or (isinstance(vol_raw, float) and math.isnan(vol_raw)):
            vol_ = None
        else:
            vol_ = int(vol_raw)

        daily_rows.append([cid, trade_date, open_, high_, low_, close_, adj_close_, vol_])

    time.sleep(0.10)

print("Downloaded row count:", len(daily_rows))

## 9. Convert downloaded rows into a structured daily price table

This cell converts the collected row list into a DataFrame and assigns standardized column names.

In [ ]:
daily_price_out = pd.DataFrame(
    daily_rows,
    columns=["company_id", "date", "open", "high", "low", "close", "adj_close", "volume"]
)

print("Daily price table shape (raw):", daily_price_out.shape)
display(daily_price_out.head())

## 10. Clean and standardize the daily price table

This cell converts numeric fields, removes duplicates, sorts the table, and ensures the final daily price dataset is ready for downstream use.

In [ ]:
for col in ["open", "high", "low", "close", "adj_close"]:
    daily_price_out[col] = pd.to_numeric(daily_price_out[col], errors="coerce").round(4)

daily_price_out["volume"] = pd.to_numeric(
    daily_price_out["volume"], errors="coerce"
).fillna(0).astype("int64")

daily_price_out = daily_price_out.sort_values(["company_id", "date"])
daily_price_out = daily_price_out.drop_duplicates(subset=["company_id", "date"], keep="last")

print("Daily price table shape (clean):", daily_price_out.shape)
display(daily_price_out.head())

## 11. Export the final daily price table

This cell exports the cleaned historical market data to `daily_price.csv`, which will be used by the clustering and Y-label notebooks.

In [ ]:
daily_price_out.to_csv("daily_price.csv", header=False, index=False, float_format="%.4f")
print("Saved: daily_price.csv")

## 12. Summary of exported intermediate files

This cell summarizes the outputs produced by this notebook and explains how they connect to downstream modules.

In [ ]:
print("Pipeline complete.")
print("Generated files:")
print("- base_universe_from_sec.csv")
print("- company.csv")
print("- daily_price.csv")